# TopiOCQA IterCQR Baseline

This notebook evaluates the frozen TopiOCQA-trained IterCQR checkpoint on all 2,514 TopiOCQA development queries. The thesis calls the history-aware baseline at budget $B=512$ route $I_{512}$. Sparse/BM25 runs locally using the TopiOCQA Lucene index. Dense/ANCE runs on one sufficiently large CUDA device.

### Local checkout setup

Use this public checkout on a local machine or a prepared Colab runtime. The next cell locates it; it does not clone a private repository, mount Drive, or request credentials.

In [ ]:
from pathlib import Path
import sys
import os

# Use an existing public checkout, locally or in Colab.
# No private repository, deploy key, or Drive mount is required.
ROOT = Path(os.environ.get("ROCC_ROOT", os.environ.get("MASTER_THESIS_PROJECT_ROOT", Path.cwd()))).resolve()
if not (ROOT / "experiments" / "rocc").exists():
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "experiments" / "rocc").exists())
os.environ["ROCC_ROOT"] = str(ROOT)
os.environ["MASTER_THESIS_PROJECT_ROOT"] = str(ROOT)
sys.path.insert(0, str(ROOT / "experiments"))
print("project:", ROOT)


In [3]:
!pip install -q pytorch-crf==0.7.2

### Runtime setup

Locates the `experiments` directory, exposes the ROCC API, and initializes the local or Colab runtime.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

EXPERIMENTS_DIR = ROOT / "experiments"
if str(EXPERIMENTS_DIR) not in sys.path:
    sys.path.insert(0, str(EXPERIMENTS_DIR))

import pandas as pd
import torch
from IPython.display import display

import rocc
from rocc.paths import project_path

pd.set_option("display.max_colwidth", 180)
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print("experiments_dir:", EXPERIMENTS_DIR)
print("device:", DEVICE)


### IterCQR runtime resources

Reuses the local TopiOCQA corpus and IterCQR checkpoint. In Colab, the helpers can mount Google Drive and link resources found there when local files are absent. The displayed status reports resource availability and checkpoint completeness.

In [5]:
rocc.prepare_colab_itercqr_runtime()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


ColabIterCQRRuntimeResult(corpus=ColabTopiOCQACorpusResult(is_colab=True, mounted=True, drive_mount=PosixPath('/content/drive'), corpora_dir=PosixPath('/content/drive/MyDrive/corpora'), drive_corpus=PosixPath('/content/drive/MyDrive/corpora/full_wiki_segments.tsv'), local_corpus=PosixPath('<original-checkout>/experiments/data/topiocqa/downloads/data/wikipedia_split/full_wiki_segments.tsv'), status='linked', size_bytes=12825331146), checkpoint=ColabIterCQRCheckpointResult(is_colab=True, mounted=True, drive_mount=PosixPath('/content/drive'), drive_checkpoint=PosixPath('/content/drive/MyDrive/IterCQR'), local_checkpoint=PosixPath('<original-checkout>/experiments/model/IterCQR/IterCQR Model'), status='linked', missing_files=(), size_bytes=892500921))

### TopiOCQA IterCQR data

Creates train and development dataloaders with batch size 16. With `include_history=False`, each input contains only the current query. The first development batch verifies sample IDs, questions, sequence lengths, and tensor shape.

In [ ]:
data = rocc.load_itercqr_dataloaders(
    dataset_name="topiocqa",
    splits=("dev","train"),
    batch_size=16,
    progress=True,
    include_history=False,
    topiocqa_download_corpus=False,
)

dev_loader = data.dataloader("dev")
first_batch = next(iter(dev_loader))

print("dataset:", data.dataset_name)
print("resource_root:", data.resources.root)
print("dev_turns:", len(dev_loader.dataset))
print("dev_batches:", len(dev_loader))
print("input_shape:", tuple(first_batch["bt_input_ids"].shape))

display(pd.DataFrame({
    "sample_id": first_batch["bt_sample_ids"],
    "question": first_batch["bt_questions"],
    "input_length": first_batch["bt_input_lengths"].tolist(),
}))


The TopiOCQA development split contains **2,514 turns** grouped into **158 dynamically padded batches** of up to 16 queries. The first batch contains the shortest current-query-only inputs with lengths of four or five tokens. Questions such as “how”, “why”, and “by whom” illustrate their strong dependence on conversational history.

### IterCQR dataset diagnostics

`analyze_itercqr_dataset()` summarizes the complete TopiOCQA train and development splits together with their actual dataloaders. It reports conversation structure, batch counts, dynamic-padding savings, input-level truncation, and unknown-token frequencies.

The segment analysis separates raw tokens, segment-budgeted tokens, and tokens retained under the global input limit.

In [7]:
analysis = rocc.analyze_itercqr_dataset(data, formatted_sample_size=10)

display(analysis.split_summary)
display(analysis.dataloader_summary)
display(analysis.padding_report)
display(analysis.truncation_report)
display(analysis.unknown_token_report)
display(analysis.segment_bucket_report)
display(analysis.segment_bucket_samples)
display(analysis.formatted_input_samples)

,split,conversations,turns,longest_conversation_turns,shortest_conversation_turns,mean_turns_per_conversation,min_topics_per_conversation,max_topics_per_conversation,mean_topics_per_conversation,min_topic_switches_per_conversation,max_topic_switches_per_conversation,mean_topic_switches_per_conversation,turns_without_positive_ctx_passage_ids,positive_ctxs_checked,missing_wiki_passages,exact_text_matches,non_exact_text_matches,all_positive_ctx_texts_exact_match
0,all,3714,47964,25,5,12.91,1,16,3.86,0,15,3.48,0,0,0,0,0,None
1,dev,205,2514,16,10,12.26,1,8,3.66,0,8,3.28,0,0,0,0,0,None
2,train,3509,45450,25,5,12.95,1,16,3.87,0,15,3.49,0,0,0,0,0,None


,split,num_turns,num_batches,batch_size,num_workers
0,dev,2514,158,16,0
1,train,45450,2841,16,0


,split,num_turns,batch_size,max_input_tokens,length_min,length_mean,length_median,length_max,real_tokens,static_total_tokens,static_pad_tokens,static_pad_fraction,bucketed_dynamic_total_tokens,bucketed_dynamic_pad_tokens,bucketed_dynamic_pad_fraction,padding_fraction_saved_points,padding_tokens_saved
0,dev,2514,16,512,4,10.806683,10.0,30,27168,1287168,1260000,0.978893,27388,220,0.008033,0.970860,1259780
1,train,45450,16,512,4,10.783432,10.0,32,490107,23270400,22780293,0.978939,490352,245,0.000500,0.978439,22780048


,split,num_turns,max_input_tokens,raw_length_min,raw_length_mean,raw_length_median,raw_length_max,final_length_min,final_length_mean,final_length_median,final_length_max,truncated_turns,truncation_rate,truncated_tokens_total,mean_truncated_tokens_all,mean_truncated_tokens_truncated_only
0,dev,2514,512,4,10.806683,10.0,30,4,10.806683,10.0,30,0,0.00000,0,0.000000,0.0
1,train,45450,512,4,10.784026,10.0,41,4,10.783432,10.0,32,10,0.00022,27,0.000594,2.7


,split,unk_token_id,unk_tokens,unk_rate,examples_with_unk,examples_with_unk_fraction
0,dev,2,1,0.000037,1,0.000398
1,train,2,5,0.000010,5,0.000110


,split,segment_type,budget_tokens,count,raw_tokens_min,raw_tokens_mean,raw_tokens_max,bucket_tokens_min,bucket_tokens_mean,bucket_tokens_max,included_tokens_min,included_tokens_mean,included_tokens_max,bucket_utilization_mean,bucket_utilization_max,bucket_truncation_rate,global_truncation_rate
0,dev,current_query,32,2514,4,10.806683,30,4,10.806683,30,4,10.806683,30,0.337709,0.9375,0.00000,0.0
1,train,current_query,32,45450,4,10.784026,41,4,10.783432,32,4,10.783432,32,0.336982,1.0000,0.00022,0.0


,split,sample_id,conv_id,turn_id,segment_index,segment_type,budget_tokens,raw_tokens,bucket_tokens,included_tokens,bucket_truncated_tokens,global_truncated_tokens,bucket_utilization,included_utilization
0,dev,dev:1:1,1,1,0,current_query,32,16,16,16,0,0,0.50000,0.50000
1,dev,dev:1:2,1,2,0,current_query,32,8,8,8,0,0,0.25000,0.25000
2,dev,dev:1:3,1,3,0,current_query,32,13,13,13,0,0,0.40625,0.40625
3,dev,dev:1:4,1,4,0,current_query,32,7,7,7,0,0,0.21875,0.21875
4,dev,dev:1:5,1,5,0,current_query,32,15,15,15,0,0,0.46875,0.46875
5,dev,dev:1:6,1,6,0,current_query,32,12,12,12,0,0,0.37500,0.37500
6,dev,dev:1:7,1,7,0,current_query,32,9,9,9,0,0,0.28125,0.28125
7,dev,dev:1:8,1,8,0,current_query,32,14,14,14,0,0,0.43750,0.43750
8,dev,dev:1:9,1,9,0,current_query,32,12,12,12,0,0,0.37500,0.37500
9,dev,dev:1:10,1,10,0,current_query,32,10,10,10,0,0,0.31250,0.31250


,split,sample_id,conv_id,turn_id,topic_document,topic_id,topic_switch,topic_switch_depth,topics_in_conversation,question,...,final_input_length,was_truncated,truncated_tokens,current_query_raw_tokens,current_query_bucket_tokens,history_segments_used,history_bucket_utilization_mean,history_bucket_utilization_max,formatted_input_text,positive_ctx_passage_ids
0,dev,dev:1:1,1,1,Dunkirk (2017 film),1,False,0,4,when will the new dunkirk film be released on dvd,...,16,False,0,16,16,0,0.0,0.0,question: when will the new dunkirk film be released on dvd,[14979903]
1,dev,dev:1:2,1,2,Dunkirk (2017 film),1,False,0,4,what is this film about,...,8,False,0,8,8,0,0.0,0.0,question: what is this film about,[14979882]
2,dev,dev:1:3,1,3,Dunkirk (2017 film),1,False,0,4,can you mention a few members of the cast,...,13,False,0,13,13,0,0.0,0.0,question: can you mention a few members of the cast,[14979882]
3,dev,dev:1:4,1,4,Dunkirk (2017 film),1,False,0,4,where was it shot,...,7,False,0,7,7,0,0.0,0.0,question: where was it shot,[14979895]
4,dev,dev:1:5,1,5,Urk,2,True,1,4,can you tell me anything about the latter place's past,...,15,False,0,15,15,0,0.0,0.0,question: can you tell me anything about the latter place's past,[25289152]
5,dev,dev:1:6,1,6,Urk,2,False,1,4,what does the economy of this place depend upon,...,12,False,0,12,12,0,0.0,0.0,question: what does the economy of this place depend upon,[25289154]
6,dev,dev:1:7,1,7,Urk,2,False,1,4,is there any waterbody nearby,...,9,False,0,9,9,0,0.0,0.0,question: is there any waterbody nearby,[25289154]
7,dev,dev:1:8,1,8,North Sea,3,True,2,4,what kind of marine organisms are found in this sea,...,14,False,0,14,14,0,0.0,0.0,question: what kind of marine organisms are found in this sea,[25631905]
8,dev,dev:1:9,1,9,North Sea,3,False,2,4,what kind of minerals are found in this sea,...,12,False,0,12,12,0,0.0,0.0,question: what kind of minerals are found in this sea,[25631927]
9,dev,dev:1:10,1,10,North Sea,3,False,2,4,does this place experiences any natural hazards,...,10,False,0,10,10,0,0.0,0.0,question: does this place experiences any natural hazards,[25631898]


The analysis covers **3,714 conversations and 47,964 turns**. Conversations contain 12.91 turns, 3.86 topics, and 3.48 topic switches on average. Every turn has at least one gold passage ID.

Because `include_history=False`, every example contains only the current query. Mean input length is approximately **10.8 tokens**. Static padding to 512 tokens would waste about **97.9%** of the input tensor, while length-bucketed dynamic padding reduces this to **0.80% on Dev** and **0.05% on Train**.

No development query is truncated. Only **10 of 45,450 training queries** exceed the 32-token query limit. Unknown tokens occur in one development and five training examples, making their frequency negligible. The formatted samples also confirm that no history segments are included.

### Validate local IterCQR assets

Provide the author checkpoint at the documented local model path. The original `t5-base` backbone can be downloaded on first use; inference uses the fine-tuned IterCQR checkpoint.

In [8]:
if rocc.is_google_colab():
    checkpoint_result = rocc.prepare_colab_itercqr_checkpoint()
    display(pd.DataFrame([checkpoint_result.to_dict()]))

t5_backbone_dir = rocc.ensure_hf_t5_backbone("t5-base")
itercqr_checkpoint_dir = rocc.validate_itercqr_checkpoint()

print("hf_t5_backbone_dir:", t5_backbone_dir)
print("itercqr_checkpoint_dir:", itercqr_checkpoint_dir)
print("itercqr_checkpoint_files:", rocc.iter_checkpoint_files())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,is_colab,mounted,drive_mount,drive_checkpoint,local_checkpoint,status,complete,missing_files,size_gib
0,True,True,/content/drive,/content/drive/MyDrive/IterCQR,/experiments/model/IterCQR/IterCQR Model,already linked,True,[],0.83


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


hf_t5_backbone_dir: <original-checkout>/experiments/model/hf/t5-base
itercqr_checkpoint_dir: <original-checkout>/experiments/model/IterCQR/IterCQR Model
itercqr_checkpoint_files: ['config.json', 'generation_config.json', 'pytorch_model.bin', 'special_tokens_map.json', 'spiece.model', 'tokenizer_config.json']


### IterCQR inference check

Loads the frozen IterCQR model and tokenizer on the selected device. One development batch is rewritten with deterministic greedy decoding and displayed alongside the original questions and gold passage IDs.

In [9]:
itercqr_rewriter = rocc.load_itercqr_rewriter(device=DEVICE)
print("model:", type(itercqr_rewriter.model).__name__)
print("tokenizer:", type(itercqr_rewriter.tokenizer).__name__)
print("device:", itercqr_rewriter.device)

rewrite_df = itercqr_rewriter.rewrite_dataloader(dev_loader, max_batches=1)
display(rewrite_df[["sample_id", "question", "rewrite", "positive_ctx_passage_ids"]])

[transformers] The following generation flags are not valid and may be ignored: ['length_penalty']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


model: T5ForConditionalGeneration
tokenizer: T5Tokenizer
device: cuda:0


,sample_id,question,rewrite,positive_ctx_passage_ids
0,dev:13:16,how,"how to use the term ""method"" used to refer to the process of creating the document ""The Way of Life"" in the context of the United States",[1578190]
1,dev:20:5,why,"what is the reason for the existence of the company ""The Company"" in the early 1900s and early 1900s based in the United States and Europe",[7771165]
2,dev:37:6,where,"where is the location of the site where the film ""The Simpsons"" was released based on the novel ""The Simpsons"" starring the character",[12462333]
3,dev:90:2,where,"where is the location of the site where the film ""The Simpsons"" was released based on the novel ""The Simpsons"" starring the character",[10854643]
4,dev:106:4,when,when if you are interested in learning more about the history of the United States or Canada during the early 1900s and early 1900s if you are,[16142705]
5,dev:110:7,why,"what is the reason for the existence of the company ""The Company"" in the early 1900s and early 1900s based in the United States and Europe",[19601104]
6,dev:196:3,when,when if you are interested in learning more about the history of the United States or Canada during the early 1900s and early 1900s if you are,[19576245]
7,dev:196:4,where,"where is the location of the site where the film ""The Simpsons"" was released based on the novel ""The Simpsons"" starring the character",[19576245]
8,dev:2:9,from where,"from where if you are interested in learning more about the history of the United States or Canada in the early 1600s and early 1900s, please contact",[7734452]
9,dev:23:5,which bank,which bank if you are interested in learning more about the history of the United States or Canada in the early 1900s and early 1900s if you,[21469230]


The current-query-only inputs are highly underspecified. Without dialogue history, IterCQR cannot resolve references such as “how”, “where”, or “by whom” and instead relies on generic decoder priors. This produces repeated and unsupported entities such as *The Simpsons*, *The Company*, and *the United States*.

The incomplete endings reflect the fixed 32-token generation limit. This sanity check demonstrates why conversational context is required. It does not represent the normal history-aware IterCQR baseline or its retrieval quality.

### Retrieval dependencies

Use Java 21 and Pyserini 2.0. The compatibility helper reports the installed environment; see the README for setup.

In [10]:
retrieval_deps = rocc.ensure_colab_retrieval_dependencies()
display(pd.DataFrame([retrieval_deps.to_dict()]))

,is_colab,installed,pyserini_available,java_home,java_available,packages,commands_run
0,True,True,True,/usr/lib/jvm/java-21-openjdk-amd64,True,[pyserini==2.0.0],[/usr/bin/python3 -m pip install pyserini==2.0.0]


### TopiOCQA BM25 runtime

Checks the Lucene index against the expected **25,700,592 passages** before loading the retriever with the fixed TopiOCQA parameters \(k_1=0.9\) and \(b=0.4\). Retrieval is skipped when the local index is unavailable.

In [9]:
TOPIOCQA_BM25_INDEX_DIR = project_path(
    "experiments",
    "data",
    "pyserini_bm25_lucene_topiocqa",
    "lucene_index",
)
TOPIOCQA_BM25_LOG_FILE = project_path(
    "experiments",
    "logs",
    "topiocqa_bm25_lucene_index.log",
)
TOPIOCQA_BM25_K1 = 0.9
TOPIOCQA_BM25_B = 0.4

topiocqa_bm25_status = rocc.inspect_bm25_lucene_index(
    TOPIOCQA_BM25_INDEX_DIR,
    source_passages=rocc.TOPIOCQA_CORPUS_PASSAGES,
    log_file=TOPIOCQA_BM25_LOG_FILE,
)
display(pd.DataFrame([topiocqa_bm25_status.to_dict()]))

if topiocqa_bm25_status.loadable:
    bm25_retriever = rocc.load_bm25_retriever(
        index_dir=TOPIOCQA_BM25_INDEX_DIR,
        k1=TOPIOCQA_BM25_K1,
        b=TOPIOCQA_BM25_B,
    )
    print("TopiOCQA BM25 Lucene index loaded:", bm25_retriever.index_dir)
else:
    bm25_retriever = None
    print("TopiOCQA BM25 skipped: Lucene index is not local/loadable. This is expected on Colab unless the index was copied there.")


,index_dir,exists,loadable,complete,num_docs,source_passages,log_file,log_exit_code,log_indexing_complete,log_indexed_docs,log_empty_docs,error
0,/experiments/data/pyserini_bm25_lucene_topiocqa/lucene_index,True,True,True,25700592,25700592,/experiments/logs/topiocqa_bm25_lucene_index.log,None,False,None,None,None


TopiOCQA BM25 Lucene index loaded: <original-checkout>/experiments/data/pyserini_bm25_lucene_topiocqa/lucene_index



Aug 20, 2026 8:34:14 AM org.apache.lucene.internal.vectorization.PanamaVectorizationProvider <init>
INFO: Java vector incubator API enabled; uses preferredBitSize=256; FMA enabled


### BM25 smoke retrieval

Retrieves the top five BM25 passages for the first generated IterCQR rewrite. Raw passage content supports qualitative inspection, while min-max normalization makes scores comparable within the returned ranking.

In [10]:
if bm25_retriever is not None:
    query = rewrite_df.iloc[0]["rewrite"]
    bm25_hits = bm25_retriever.search(query, top_k=5, include_raw=True)
    bm25_hits_norm = rocc.normalize_retrieval_scores(bm25_hits, method="minmax")
    display(pd.DataFrame(bm25_hits_norm))
else:
    print("BM25 smoke retrieval skipped: bm25_retriever is not loaded.")


,rank,docid,score,title,text,raw_passage_id,score_norm
0,1,6413180,21.223200,Method (patent) [SEP] Introduction,"In United States patent law, a method, also called ""process"", is one of the four principal categories of things that may be patented through ""utility patents"". The other three ...",None,1.000000
1,2,20590752,19.420000,Darwin Core [SEP] Description,"The Darwin Core is a body of standards. It includes a glossary of terms (in other contexts these might be called properties, elements, fields, columns, attributes, or concepts)...",None,0.457799
2,3,19277447,19.240499,SEXINT [SEP] Uses,"The term was first used specifically in reference to the practice by Five Eyes member, the National Security Agency of the United States of America. It is unclear how often the...",None,0.403825
3,4,6965984,18.743700,Rotational molding [SEP] History,"In 1855, R. Peters of Britain documented the first use of biaxial rotation and heat. This rotational molding process was used to create metal artillery shells and other hollow ...",None,0.254443
4,5,5489851,17.897499,Indigenization [SEP] Linguistics,"In this context, indigenization is used to refer to how a language is adopted in a certain area such as French in Africa. The term is used to describe the process of conveying ...",None,0.000000


The current-query-only rewrite drives BM25 toward passages containing generic terms such as “method”, “process”, and “United States”. The highest-ranked passage is therefore lexically compatible but unrelated to the intended conversational context. The expected gold passage is absent from the displayed top five, illustrating how an ungrounded rewrite propagates directly into retrieval failure. Normalized scores are provided only for within-list comparison and do not alter the ranking.

### Validate the local ANCE index

Place the complete index locally before running; no automatic remote download occurs. Completed shards are summarized and checked against available GPU memory with a 4 GiB reserve. Retrieval retains float32 vectors, 50,000-passage chunks and query batches of 64.

In [ ]:
DENSE_INDEX_DIR = project_path(
    "experiments",
    "data",
    "pyserini_ance_faiss_topiocqa",
    "faiss_flat_index_full_sharded",
)
DENSE_AZURE_SAS_FILE = None
DENSE_INDEX_DTYPE = "float32"
DENSE_CHUNK_SIZE = 50_000
DENSE_QUERY_BATCH_SIZE = 64

try:
    dense_summary = rocc.summarize_dense_faiss_shards(DENSE_INDEX_DIR)
    DENSE_INDEX_DIR = dense_summary.index_dir
except (FileNotFoundError, RuntimeError, ValueError, KeyError):
    dense_summary = None

if dense_summary is not None and (not dense_summary.contiguous or dense_summary.num_passages != rocc.TOPIOCQA_CORPUS_PASSAGES):
    dense_summary = None
if dense_summary is None:
    print("Local TopiOCQA ANCE index is missing/incomplete; finish copying or building it first.")

if dense_summary is not None:
    DENSE_INDEX_DIR = dense_summary.index_dir
    display(pd.DataFrame([dense_summary.to_dict()]))

    if DEVICE.startswith("cuda"):
        dense_capacity = rocc.check_dense_gpu_capacity(
            dense_summary,
            device=DEVICE,
            index_dtype=DENSE_INDEX_DTYPE,
            reserve_gib=4.0,
        )
        display(pd.DataFrame([dense_capacity.__dict__]))
    else:
        dense_capacity = None
        print("CUDA is not active; full ANCE Torch search requires a sufficiently large CUDA device.")
else:
    dense_summary = None
    dense_capacity = None


### ANCE retriever loading

Loads the complete sharded ANCE index onto the CUDA device only when the index is available and the GPU-memory check succeeds. Vectors are searched with the configured data type, chunk size, and query batch size. A five-passage search for “Eliza Fletcher” verifies that the retriever is operational. Loading progress is written to `/content/ance_index_load_status.log`.

In [ ]:
dense_retriever = None

if DENSE_INDEX_DIR.exists() and DEVICE.startswith("cuda") and dense_capacity is not None and dense_capacity.fits:
    dense_retriever = rocc.load_dense_ance_torch_retriever(
        index_dir=DENSE_INDEX_DIR,
        device=DEVICE,
        index_dtype=DENSE_INDEX_DTYPE,
        chunk_size=DENSE_CHUNK_SIZE,
        query_batch_size=DENSE_QUERY_BATCH_SIZE,
        progress=True,
        status_log=project_path("experiments", "logs", "ance_index_load_status.log"),
    )
    display(pd.DataFrame(dense_retriever.search("Eliza Fletcher", top_k=5)))
else:
    print("Dense Full Retriever not loaded: requires a local index, CUDA, and sufficient free VRAM.")


### TopiOCQA B512 baseline configuration

Evaluates all **2,514 TopiOCQA development queries** with route $I_{512}$. The current query $q_t$ comes first, followed by historical answers and queries from $H_t$ in most-recent-first order. Each field is limited to 32 T5 tokens before the total input budget $B=512$ is applied, so the serialized input length satisfies $n\leq B$. Prefixes and special tokens count towards these limits. Greedy decoding has maximum output sequence length $m_{\max}=32$.

Retrieval depth is $k_{\mathrm{ret}}=100$. The reported metrics are $\mathrm{MRR}$ over those retrieved passages, $\mathrm{nDCG}@3$, $\mathrm{Recall}@10$, and $\mathrm{Recall}@100$. Their 95% confidence intervals use $N_{\mathrm{boot}}=10{,}000$ query-level bootstrap resamples. The later final retrieval experiments use depth 1,000 instead.

In [ ]:
RETRIEVAL_TOP_K = 100
RETRIEVAL_MAX_BATCHES = None
EVAL_KS = (3, 10, 100)
BOOTSTRAP_SAMPLES = 10000
INCLUDE_HISTORY = True
TOPIOCQA_BASELINE_BUDGETS = (512,)

print("expected dev queries:", len(dev_loader.dataset))
print("include_history:", INCLUDE_HISTORY)
print("baseline budgets:", TOPIOCQA_BASELINE_BUDGETS)

expected dev queries: 2514
include_history: True
baseline budgets: (512,)


### TopiOCQA BM25 B512 baseline

Runs the canonical IterCQR baseline over all 2,514 TopiOCQA development queries. Recency-ordered dialogue history is included within the global 512-token input budget. Each rewrite is evaluated against BM25 using the top 100 passages. Aggregate metrics, bootstrap confidence intervals, and per-query results are persisted under `full_pipeline_runs/topiocqa_dev_sparse_itercqr_max512`.

In [22]:
topiocqa_sparse_budget_sweep = None
if bm25_retriever is not None:
    topiocqa_sparse_budget_sweep = rocc.run_itercqr_budget_sweep(
        dataset_name="topiocqa",
        split="dev",
        retriever_kind="sparse",
        budgets=TOPIOCQA_BASELINE_BUDGETS,
        bm25_index_dir=TOPIOCQA_BM25_INDEX_DIR,
        run_id="topiocqa_dev_sparse_itercqr",
        batch_size=16,
        top_k=RETRIEVAL_TOP_K,
        eval_ks=EVAL_KS,
        bootstrap_samples=BOOTSTRAP_SAMPLES,
        max_batches=RETRIEVAL_MAX_BATCHES,
        device=DEVICE,
        progress=True,
        rewriter=itercqr_rewriter,
        retriever=bm25_retriever,
        include_history=INCLUDE_HISTORY,
    )
    display(topiocqa_sparse_budget_sweep.summary)
    display(topiocqa_sparse_budget_sweep.evaluation_aggregate)
else:
    print("Sparse budget sweep skipped: bm25_retriever is not loaded.")

[1/4] Prepare shared dataset resources
[2/4] Use shared IterCQR rewriter
[3/4] Use shared retriever index
[4/4] Use shared retriever
[1/4] Load dataset split for budget


[2/4] Run rewrite + retrieval


[3/4] Evaluate retrieval
[4/4] Save eval results


,budget_tokens,include_history,run_id,dataset_name,split,retriever_kind,qrecc_dense_mode,MRR,nDCG@3,R@3,nDCG@10,R@10,nDCG@100,R@100
0,512,True,topiocqa_dev_sparse_itercqr_max512,topiocqa,dev,sparse,run_partial,0.16473,0.150811,0.188146,0.188421,0.293158,0.236777,0.531026


,budget_tokens,include_history,metric,n,mean,variance
0,512,True,MRR,2514,0.164730,0.092874
1,512,True,nDCG@3,2514,0.150811,0.106559
2,512,True,R@3,2514,0.188146,0.152808
3,512,True,nDCG@10,2514,0.188421,0.107514
4,512,True,R@10,2514,0.293158,0.207299
5,512,True,nDCG@100,2514,0.236777,0.097113
6,512,True,R@100,2514,0.531026,0.249136


### TopiOCQA ANCE B512 baseline

Runs the canonical IterCQR baseline over all 2,514 TopiOCQA development queries using the dense ANCE retriever. Recency-ordered dialogue history is included within the global 512-token input budget. Each rewrite retrieves the top 100 passages. Aggregate metrics, bootstrap confidence intervals, and per-query results are persisted under `full_pipeline_runs/topiocqa_dev_dense_itercqr_max512`.

In [14]:
topiocqa_dense_budget_sweep = None
if dense_retriever is not None:
    topiocqa_dense_budget_sweep = rocc.run_itercqr_budget_sweep(
        dataset_name="topiocqa",
        split="dev",
        retriever_kind="dense",
        budgets=TOPIOCQA_BASELINE_BUDGETS,
        dense_index_dir=DENSE_INDEX_DIR,
        azure_sas_file=DENSE_AZURE_SAS_FILE,
        run_id="topiocqa_dev_dense_itercqr",
        batch_size=16,
        top_k=RETRIEVAL_TOP_K,
        eval_ks=EVAL_KS,
        bootstrap_samples=BOOTSTRAP_SAMPLES,
        max_batches=RETRIEVAL_MAX_BATCHES,
        device=DEVICE,
        progress=True,
        rewriter=itercqr_rewriter,
        retriever=dense_retriever,
        include_history=INCLUDE_HISTORY,
    )
    display(topiocqa_dense_budget_sweep.summary)
    display(topiocqa_dense_budget_sweep.evaluation_aggregate)
else:
    print("Dense budget sweep skipped: dense_retriever is not loaded.")

[1/4] Prepare shared dataset resources
[2/4] Use shared IterCQR rewriter
[3/4] Use shared retriever index
[4/4] Use shared retriever
[1/4] Load dataset split for budget


[2/4] Run rewrite + retrieval


[3/4] Evaluate retrieval
[4/4] Save eval results


,budget_tokens,include_history,run_id,dataset_name,split,retriever_kind,qrecc_dense_mode,MRR,nDCG@3,R@3,nDCG@10,R@10,nDCG@100,R@100
0,512,True,topiocqa_dev_dense_itercqr_max512,topiocqa,dev,dense,run_partial,0.254555,0.241674,0.292363,0.287204,0.418059,0.328777,0.614956


,budget_tokens,include_history,metric,n,mean,variance
0,512,True,MRR,2514,0.254555,0.135428
1,512,True,nDCG@3,2514,0.241674,0.154524
2,512,True,R@3,2514,0.292363,0.206969
3,512,True,nDCG@10,2514,0.287204,0.147212
4,512,True,R@10,2514,0.418059,0.243382
5,512,True,nDCG@100,2514,0.328777,0.130674
6,512,True,R@100,2514,0.614956,0.236879


### BM25 bootstrap summary

Reports bootstrap estimates and 95% confidence intervals for the TopiOCQA BM25 $I_{512}$ baseline. The summary is available after the sparse baseline run has completed successfully.

In [23]:
if topiocqa_sparse_budget_sweep is not None:
    display(topiocqa_sparse_budget_sweep.bootstrap_summary)
else:
    print("Sparse evaluation summary skipped: topiocqa_sparse_budget_sweep is unavailable.")

,budget_tokens,include_history,metric,bootstrap_samples,confidence,mean,variance,ci95_low,ci95_high
0,512,True,MRR,10000,0.95,0.164876,0.000036,0.153151,0.176796
1,512,True,nDCG@3,10000,0.95,0.150983,0.000042,0.138490,0.163587
2,512,True,R@3,10000,0.95,0.188359,0.000060,0.173031,0.203660
3,512,True,nDCG@10,10000,0.95,0.188568,0.000042,0.175896,0.201380
4,512,True,R@10,10000,0.95,0.293288,0.000081,0.275259,0.310670
5,512,True,nDCG@100,10000,0.95,0.236918,0.000038,0.224983,0.249122
6,512,True,R@100,10000,0.95,0.531150,0.000098,0.511933,0.550527


### ANCE bootstrap summary

Reports bootstrap estimates and 95% confidence intervals for the TopiOCQA ANCE $I_{512}$ baseline. The summary is available after the dense baseline run has completed successfully.

In [15]:
if topiocqa_dense_budget_sweep is not None:
    display(topiocqa_dense_budget_sweep.bootstrap_summary)
else:
    print("Dense evaluation summary skipped: topiocqa_dense_budget_sweep is unavailable.")

,budget_tokens,include_history,metric,bootstrap_samples,confidence,mean,variance,ci95_low,ci95_high
0,512,True,MRR,10000,0.95,0.254684,0.000054,0.240388,0.269144
1,512,True,nDCG@3,10000,0.95,0.241818,0.000062,0.226533,0.257225
2,512,True,R@3,10000,0.95,0.292520,0.000084,0.274861,0.310263
3,512,True,nDCG@10,10000,0.95,0.287339,0.000059,0.272260,0.302311
4,512,True,R@10,10000,0.95,0.418183,0.000098,0.398568,0.437152
5,512,True,nDCG@100,10000,0.95,0.328872,0.000052,0.314698,0.342949
6,512,True,R@100,10000,0.95,0.614892,0.000094,0.595863,0.633652
